# Step 2(进阶数据读取）: Parser 与单路径解码器

这一节对照两套更精细的数据读取入口，适合用于ms级数据索引：

- `TianmoucDataReader(..., use_data_parser=True)`：返回 `TianmoucDataSampleParser`，适合用方法访问 sample。
- `cone_tmdat_to_npy / rod_tmdat_to_npy / multiple_rods_tmdat_to_npy`：直接解码单个 `.tmdat`，适合底层检查。

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'tianmoucv').exists() and (candidate / 'tianmoucv_example').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError('找不到 tianmoucv 仓库根目录，请从仓库内打开本 Notebook。')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import torch


In [ ]:
from tianmoucv.data import (
    TianmoucDataReader,
    cone_tmdat_to_npy,
    multiple_rods_tmdat_to_npy,
    rod_tmdat_to_npy,
)
from tianmoucv.utils import ensure_tutorial_dataset

info = ensure_tutorial_dataset()

## 1. 通过 `use_data_parser=True` 返回结构化 parser

In [ ]:
dataset = TianmoucDataReader(
    str(info.sample_dir),
    N=2,
    print_info=False,
    use_data_parser=True,
)
parsed = dataset[0]
parsed.help()

In [ ]:
print('dataRate =', parsed.get_data_rate())
print('RGB[0] shape =', parsed.get_rgb(0).shape)
print('HDR[0] shape =', parsed.get_hdr_fusion(0).shape)
print('TSD upsampled shape =', parsed.get_tsd(0, ifUpSampled=True).shape)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(parsed.get_rgb(0))
plt.title('parsed.get_rgb(0)')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(parsed.get_hdr_fusion(0))
plt.title('parsed.get_hdr_fusion(0)')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. 直接解码 `.tmdat`

这套接口绕过 `TianmoucDataReader` 的 sample 组织逻辑，直接读取 `cone/*.tmdat` 与 `rod/*.tmdat`。

In [ ]:
cone_file = next((info.sample_dir / 'cone').glob('*.tmdat'))
rod_file = next((info.sample_dir / 'rod').glob('*.tmdat'))

cone_rgb, cone_ts = cone_tmdat_to_npy(str(cone_file), idx=0, mode='RGB', print_info=False)
cone_raw, _ = cone_tmdat_to_npy(str(cone_file), idx=0, mode='RAW', print_info=False)
rod_tsd, rod_ts = rod_tmdat_to_npy(str(rod_file), idx=0, print_info=False)
rod_seq, rod_seq_ts = multiple_rods_tmdat_to_npy(str(rod_file), idx=0, length=4, print_info=False)

print('cone_rgb:', cone_rgb.shape, cone_ts)
print('cone_raw:', cone_raw.shape)
print('rod_tsd:', rod_tsd.shape, rod_ts)
print('rod_seq:', rod_seq.shape, rod_seq_ts[:2], '...')

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(cone_rgb)
plt.title('cone_tmdat_to_npy(..., mode="RGB")')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(cone_raw, cmap='gray')
plt.title('cone raw')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(rod_tsd[0], cmap='gray')
plt.title('rod TD')
plt.axis('off')
plt.tight_layout()
plt.show()